# benchmarking_pinball

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"
REFERENCE_CONFIG = Path("../tuned/config.py")

In [ ]:
# Load Agent0's and Agent1's stored runs per Pinball setting, and the DQN runs of
# tuned as the reference, skipping any not stored yet.
AGENT0 = "Agent0"
AGENT1 = "Agent1"
DQN = "DQN"
SETTINGS = ["empty", "box", "easy"]

spec = importlib.util.spec_from_file_location("reference_config", REFERENCE_CONFIG)
reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference)
STORES = {
    AGENT0: (EXPERIMENT, "agent0_pinball"),
    AGENT1: (EXPERIMENT, "agent1_pinball"),
    DQN: (reference.EXPERIMENT, "dqn_pinball"),
}
AGENTS = list(STORES)

results = {}
for agent, (experiment, prefix) in STORES.items():
    for setting in SETTINGS:
        runs = load_results(experiment, f"{prefix}_{setting}")
        if runs is not None:
            results[agent, setting] = runs

In [ ]:
# Compute each agent's mean return over steps per setting with a 95% bootstrap CI.
curves = {}
for (agent, setting), runs in results.items():
    steps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, setting] = (steps, mean_ci(returns))

In [ ]:
# Style for the learning-curve plots. Every piece of text shares one size.
FONT_SIZE = 32
STYLE = {
    "figure.figsize": (24, 8),
    "font.family": "Times New Roman",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE,
    "ytick.labelsize": FONT_SIZE,
    "legend.fontsize": FONT_SIZE,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
BAND_ALPHA = 0.2
AGENT0_COLOR = "tab:red"
AGENT1_COLOR = "tab:green"
DQN_COLOR = "tab:blue"
COLORS = {AGENT0: AGENT0_COLOR, AGENT1: AGENT1_COLOR, DQN: DQN_COLOR}
# Only the left panel carries the legend, since the panels share colors.
LEGEND = {
    "loc": "center right",
    "bbox_to_anchor": (1.0, 0.35),
    "handlelength": 1,
    "handletextpad": 0.4,
}
LEGEND_MARKER = {"marker": "o", "linestyle": "none", "markersize": 14}
BOX_ASPECT = 1
# Ticks sit only at the axis ends, so the labels can tuck in between them.
LABEL_PAD = -20
XLABEL = "Time Steps"
YLABEL = "Return"
YLIM = (-1000, 0)
YTICKS = [-1000, 0]


def title(setting):
    return f"Pinball {setting.title()}"

In [ ]:
# Plot every setting's panel side by side. Each panel has its own y-axis, so
# each is labelled. Agent0 and Agent1 are drawn over the DQN reference where
# runs are stored.
with plt.rc_context(STYLE):
    fig, axes = plt.subplots(1, len(SETTINGS))
    for ax, setting in zip(axes, SETTINGS, strict=True):
        ax.set_box_aspect(BOX_ASPECT)
        ax.set_title(title(setting))
        ax.set_xlabel(XLABEL, labelpad=LABEL_PAD)
        ax.set_ylabel(YLABEL, labelpad=LABEL_PAD)
        ax.set_ylim(YLIM)
        ax.set_yticks(YTICKS)
        for agent in AGENTS:
            if (agent, setting) not in curves:
                continue
            steps, (mean, low, high) = curves[agent, setting]
            color = COLORS[agent]
            ax.fill_between(steps, low, high, color=color, alpha=BAND_ALPHA)
            ax.plot(steps, mean, color=color)
            ax.set_xticks([0, steps[-1]], ["0", f"{steps[-1] // 1000}K"])
    fig.tight_layout()
    drawn = [a for a in AGENTS if any((a, s) in curves for s in SETTINGS)]
    handles = [Line2D([], [], color=COLORS[a], **LEGEND_MARKER) for a in drawn]
    axes[0].legend(handles, drawn, **LEGEND)
figures = {"grid": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight", dpi=200)
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")